In [2]:
from pathlib import Path
import gc

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path("data/tables")
PARTS_TO_PREDICT = ["part_2", "part_3", "part_4"]
RANDOM_STATE = 42


def add_date_features(frame):
    result = frame.copy()
    date = pd.to_datetime(result["order_datetime"])
    result["year"] = date.dt.year.astype("int16")
    result["month"] = date.dt.month.astype("int8")
    result["day"] = date.dt.day.astype("int8")
    result["day_of_week"] = date.dt.dayofweek.astype("int8")
    result["day_of_year"] = date.dt.dayofyear.astype("int16")
    result["week_of_year"] = date.dt.isocalendar().week.astype("int16")
    result["days_since_start"] = (date - pd.Timestamp("2021-01-01")).dt.days.astype("int16")
    result["amount_log"] = np.log1p(result["dollar_value"].clip(lower=0))
    return result


def entity_features(reference, target, entity_column, prefix):
    target = target.copy()
    if entity_column not in target.columns:
        target[entity_column] = -1
    if entity_column not in reference.columns:
        mean = reference["fraud_probability"].mean()
        target[f"{prefix}_mean"] = mean
        target[f"{prefix}_std"] = 0.0
        target[f"{prefix}_count"] = 0.0
        return target
    grouped = reference.groupby(entity_column)["fraud_probability"]
    summary = grouped.agg(["mean", "std", "count"]).rename(
        columns={"mean": f"{prefix}_mean", "std": f"{prefix}_std", "count": f"{prefix}_count"}
    )
    result = target.join(summary, on=entity_column)
    result[f"{prefix}_mean"] = result[f"{prefix}_mean"].fillna(reference["fraud_probability"].mean())
    result[f"{prefix}_std"] = result[f"{prefix}_std"].fillna(0)
    result[f"{prefix}_count"] = result[f"{prefix}_count"].fillna(0)
    return result


def make_features(transactions, consumer_reference, merchant_reference):
    features = add_date_features(transactions)
    features = entity_features(consumer_reference, features, "user_id", "consumer")
    features = entity_features(merchant_reference, features, "merchant_abn", "merchant")
    feature_columns = [
        "dollar_value", "amount_log", "year", "month", "day", "day_of_week",
        "day_of_year", "week_of_year", "days_since_start", "consumer_mean",
        "consumer_std", "consumer_count", "merchant_mean", "merchant_std", "merchant_count",
    ]
    return features[feature_columns].replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)


def iter_prediction_transactions():
    """Yield one transaction partition at a time to keep memory bounded."""
    for part in PARTS_TO_PREDICT:
        for partition in sorted((ROOT / part).glob("order_datetime=*")):
            parquet_files = list(partition.glob("*.parquet"))
            if not parquet_files:
                continue
            frame = pd.read_parquet(parquet_files[0])
            frame["order_datetime"] = pd.to_datetime(partition.name.split("=", 1)[1])
            yield part, frame


def make_model():
    return RandomForestRegressor(
        n_estimators=80, max_features="sqrt", min_samples_leaf=3,
        random_state=RANDOM_STATE, n_jobs=1,
    )


def train_model(label_frame, prefix):
    label_frame = label_frame.sort_values("order_datetime").reset_index(drop=True)
    split_date = label_frame["order_datetime"].quantile(0.8)
    train_labels = label_frame[label_frame["order_datetime"] < split_date]
    valid_labels = label_frame[label_frame["order_datetime"] >= split_date]

    train_features = make_features(train_labels.assign(dollar_value=0), train_labels, train_labels)
    valid_features = make_features(valid_labels.assign(dollar_value=0), train_labels, train_labels)
    validation_model = make_model()
    validation_model.fit(train_features, train_labels["fraud_probability"])
    validation_prediction = np.clip(validation_model.predict(valid_features), 0, 100)
    actual = valid_labels["fraud_probability"].to_numpy()
    errors = np.abs(actual - validation_prediction)
    report = {
        "model": prefix,
        "train_rows": len(train_labels),
        "validation_rows": len(valid_labels),
        "validation_start": valid_labels["order_datetime"].min(),
        "validation_end": valid_labels["order_datetime"].max(),
        "mae": mean_absolute_error(actual, validation_prediction),
        "rmse": mean_squared_error(actual, validation_prediction) ** 0.5,
        "r2": r2_score(actual, validation_prediction),
        "within_5_points_pct": (errors <= 5).mean() * 100,
        "within_10_points_pct": (errors <= 10).mean() * 100,
    }

    final_model = make_model()
    final_model.fit(
        make_features(label_frame.assign(dollar_value=0), label_frame, label_frame),
        label_frame["fraud_probability"],
    )
    return final_model, report


consumer_labels = pd.read_csv(ROOT / "part_1/consumer_fraud_probability.csv", parse_dates=["order_datetime"])
merchant_labels = pd.read_csv(ROOT / "part_1/merchant_fraud_probability.csv", parse_dates=["order_datetime"])
consumer_model, consumer_report = train_model(consumer_labels, "Consumer")
merchant_model, merchant_report = train_model(merchant_labels, "Merchant")

validation_report = pd.DataFrame([consumer_report, merchant_report])
validation_report.to_csv(ROOT / "model_validation_report.csv", index=False)
print("Validation report: latest 20% of labelled dates held out")
print(validation_report.to_string(index=False))

output_columns = [
    "order_id", "order_datetime", "user_id", "merchant_abn", "dollar_value",
    "consumer_fraud_probability", "merchant_fraud_probability", "fraud_probability",
]
output_path = ROOT / "part_2_4_fraud_probability_predictions.csv"
written_rows = 0
first_batch = None
for part, transactions in iter_prediction_transactions():
    prediction_features = make_features(transactions, consumer_labels, merchant_labels)
    transactions["consumer_fraud_probability"] = np.clip(consumer_model.predict(prediction_features), 0, 100)
    transactions["merchant_fraud_probability"] = np.clip(merchant_model.predict(prediction_features), 0, 100)
    transactions["fraud_probability"] = transactions[["consumer_fraud_probability", "merchant_fraud_probability"]].mean(axis=1)
    transactions[output_columns].to_csv(
        output_path, mode="w" if written_rows == 0 else "a", header=written_rows == 0, index=False
    )
    written_rows += len(transactions)
    if first_batch is None:
        first_batch = transactions[output_columns].head()
    del transactions, prediction_features
    gc.collect()

if written_rows == 0:
    raise FileNotFoundError("No Parquet transaction files were found in parts 2-4.")
print(f"Wrote {written_rows:,} predictions to {output_path}")
print(first_batch)

Validation report: latest 20% of labelled dates held out
   model  train_rows  validation_rows validation_start validation_end       mae      rmse        r2  within_5_points_pct  within_10_points_pct
Consumer       27886             6978       2022-01-01     2022-02-27  6.697629 11.375208 -0.212547            58.999713             83.691602
Merchant          89               25       2021-12-18     2022-02-27 15.243773 20.129992 -0.091977            24.000000             44.000000
Wrote 14,195,505 predictions to data/tables/part_2_4_fraud_probability_predictions.csv
                               order_id order_datetime  user_id  merchant_abn  \
0  0c37b3f7-c7f1-48cb-bcc7-0a58e76608ea     2021-02-28        1   28000487688   
1  9e18b913-0465-4fd4-92fd-66d15e65d93c     2021-02-28    18485   62191208634   
2  40a2ff69-ea34-4657-8429-df7ca957d6a1     2021-02-28        1   83690644458   
3  f4c1a5ae-5b76-40d0-ae0f-cb9730ac325a     2021-02-28    18488   39649557865   
4  cd09bdd6-f56d-489f-

In [ ]:
# Create a fraud flag using an 80% probability threshold.
predictions = pd.read_csv(output_path)
predictions["fraud_probability_normalized"] = predictions["fraud_probability"] / 100
predictions["is_fraud"] = predictions["fraud_probability_normalized"] > 0.8

flagged_output_path = ROOT / "part_2_4_fraud_probability_predictions_with_flag.csv"
predictions.to_csv(flagged_output_path, index=False)

print(f"Flagged {predictions['is_fraud'].sum():,} of {len(predictions):,} transactions as fraud")
print(f"Saved flagged predictions to {flagged_output_path}")
print(predictions[["fraud_probability", "fraud_probability_normalized", "is_fraud"]].head())